In [1]:
# Periksa file & buat konfigurasi khusus test

from pathlib import Path

# Lokasi proyek SSD MobileNet
PROJECT_DIR = Path("/home/rzangef/SSD-PPE")

CONFIG_PATH = PROJECT_DIR / "models/pipeline_ppe.config"
TEST_RECORD = PROJECT_DIR / "dataset/tfrecord/test.record"
MODEL_DIR = (
    PROJECT_DIR
    / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)
CHECKPOINT_PREFIX = MODEL_DIR / "ckpt-24"

# Konfigurasi khusus evaluasi test
TEST_CONFIG_PATH = PROJECT_DIR / "models/pipeline_ppe_test.config"

# 1. Periksa keberadaan file
required_files = {
    "Konfigurasi asli": CONFIG_PATH,
    "Data test": TEST_RECORD,
    "Checkpoint index": Path(str(CHECKPOINT_PREFIX) + ".index"),
    "Checkpoint data": Path(
        str(CHECKPOINT_PREFIX) + ".data-00000-of-00001"
    ),
}

print("=== PEMERIKSAAN FILE SSD ===")
for name, path in required_files.items():
    print(f"{name}: {'ADA' if path.exists() else 'TIDAK ADA'}")
    print(f"  {path}")

# Hentikan proses jika ada file penting yang tidak tersedia
missing = [name for name, path in required_files.items()
           if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"File berikut belum ditemukan: {missing}"
    )

# 2. Baca konfigurasi asli
config_text = CONFIG_PATH.read_text()

# Pastikan konfigurasi evaluasi awal menggunakan val.record
VAL_RECORD_PATH = str(
    PROJECT_DIR / "dataset/tfrecord/val.record"
)
TEST_RECORD_PATH = str(TEST_RECORD)

if VAL_RECORD_PATH not in config_text:
    raise ValueError(
        "Path val.record tidak ditemukan persis di konfigurasi. "
        "Periksa bagian eval_input_reader sebelum melanjutkan."
    )

# 3. Ganti path validasi dengan path test hanya pada salinan
test_config_text = config_text.replace(
    VAL_RECORD_PATH,
    TEST_RECORD_PATH
)

# Validasi agar konfigurasi training tidak ikut berubah
assert TEST_RECORD_PATH in test_config_text
assert VAL_RECORD_PATH not in test_config_text

TEST_CONFIG_PATH.write_text(test_config_text)

print("\n=== KONFIGURASI TEST BERHASIL DIBUAT ===")
print("Konfigurasi asli :", CONFIG_PATH)
print("Konfigurasi test :", TEST_CONFIG_PATH)
print("Data evaluasi    :", TEST_RECORD_PATH)
print("Konfigurasi asli tidak diubah.")

=== PEMERIKSAAN FILE SSD ===
Konfigurasi asli: ADA
  /home/rzangef/SSD-PPE/models/pipeline_ppe.config
Data test: ADA
  /home/rzangef/SSD-PPE/dataset/tfrecord/test.record
Checkpoint index: ADA
  /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-24.index
Checkpoint data: ADA
  /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-24.data-00000-of-00001

=== KONFIGURASI TEST BERHASIL DIBUAT ===
Konfigurasi asli : /home/rzangef/SSD-PPE/models/pipeline_ppe.config
Konfigurasi test : /home/rzangef/SSD-PPE/models/pipeline_ppe_test.config
Data evaluasi    : /home/rzangef/SSD-PPE/dataset/tfrecord/test.record
Konfigurasi asli tidak diubah.


In [2]:
# Jalankan evaluasi checkpoint ckpt-24

import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/home/rzangef/SSD-PPE")
RESEARCH_DIR = PROJECT_DIR / "object-detection-api/research"

SCRIPT = RESEARCH_DIR / "object_detection/model_main_tf2.py"
TEST_CONFIG_PATH = PROJECT_DIR / "models/pipeline_ppe_test.config"

MODEL_DIR = (
    PROJECT_DIR
    / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

EVAL_OUTPUT_DIR = MODEL_DIR / "eval_test_ckpt24"

# Pastikan skrip dan konfigurasi tersedia
assert SCRIPT.exists(), f"Skrip tidak ditemukan: {SCRIPT}"
assert TEST_CONFIG_PATH.exists(), (
    f"Konfigurasi tidak ditemukan: {TEST_CONFIG_PATH}"
)

cmd = [
    sys.executable,
    str(SCRIPT),
    f"--pipeline_config_path={TEST_CONFIG_PATH}",
    f"--model_dir={EVAL_OUTPUT_DIR}",
    f"--checkpoint_dir={MODEL_DIR}",
    "--eval_timeout=120",
    "--alsologtostderr",
]

print("=== MENJALANKAN EVALUASI SSD PADA TEST ===")
print("Python        :", sys.executable)
print("Checkpoint dir:", MODEL_DIR)
print("Konfigurasi   :", TEST_CONFIG_PATH)
print("Output        :", EVAL_OUTPUT_DIR)
print("\nProses dimulai. Tunggu sampai selesai...\n")

result = subprocess.run(
    cmd,
    cwd=str(RESEARCH_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=1800
)

# Tampilkan log evaluasi
print(result.stdout[-16000:])
print("\nReturn code:", result.returncode)

if result.returncode == 0:
    print("Proses evaluasi selesai tanpa error proses.")
else:
    print("Evaluasi gagal. Periksa pesan error di atas.")

=== MENJALANKAN EVALUASI SSD PADA TEST ===
Python        : /home/rzangef/SSD-PPE/.venv/bin/python
Checkpoint dir: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch
Konfigurasi   : /home/rzangef/SSD-PPE/models/pipeline_ppe_test.config
Output        : /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/eval_test_ckpt24

Proses dimulai. Tunggu sampai selesai...

2026-10-19 22:26:27.862392: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-19 22:26:28.639461: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-19 22:26:28.639952: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-19 22:26:28.793775: E e

In [3]:
# Cek versi tensorflow dan tf_slim

import sys
import importlib.metadata as metadata
import tensorflow as tf
import tensorflow.python.ops.control_flow_ops as control_flow_ops
import tf_slim

print("=== ENVIRONMENT SSD MOBILENET ===")
print("Python executable :", sys.executable)
print("TensorFlow version:", tf.__version__)
print("TensorFlow path   :", tf.__file__)
print("tf_slim path      :", tf_slim.__file__)

print("\n=== PEMERIKSAAN FUNGSI COND ===")
print(
    "control_flow_ops.cond tersedia:",
    hasattr(control_flow_ops, "cond")
)
print(
    "tf.cond tersedia:",
    hasattr(tf, "cond")
)
print(
    "tf.compat.v1.cond tersedia:",
    hasattr(tf.compat.v1, "cond")
)

print("\n=== VERSI PAKET ===")
for package in [
    "tensorflow",
    "tensorflow-cpu",
    "tf-models-official",
    "tf-slim",
    "object-detection",
    "protobuf",
    "keras",
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: metadata paket tidak ditemukan")

2026-10-19 23:18:04.945790: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-19 23:18:05.697025: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-19 23:18:05.697366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-19 23:18:05.843019: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-19 23:18:06.162165: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-19 23:18:06.169226: I tensorflow/core/platform/cpu_feature_guard.cc:1

=== ENVIRONMENT SSD MOBILENET ===
Python executable : /home/rzangef/SSD-PPE/.venv/bin/python
TensorFlow version: 2.15.1
TensorFlow path   : /home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/__init__.py
tf_slim path      : /home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/__init__.py

=== PEMERIKSAAN FUNGSI COND ===
control_flow_ops.cond tersedia: False
tf.cond tersedia: True
tf.compat.v1.cond tersedia: True

=== VERSI PAKET ===
tensorflow: 2.15.1
tensorflow-cpu: metadata paket tidak ditemukan
tf-models-official: 2.15.0
tf-slim: 1.1.0
object-detection: metadata paket tidak ditemukan
protobuf: 4.23.4
keras: 2.15.0


In [1]:
# Uji sementara evaluasi
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/home/rzangef/SSD-PPE")
RESEARCH_DIR = PROJECT_DIR / "object-detection-api/research"
SCRIPT = RESEARCH_DIR / "object_detection/model_main_tf2.py"

TEST_CONFIG = PROJECT_DIR / "models/pipeline_ppe_test.config"
MODEL_DIR = (
    PROJECT_DIR
    / "training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)
OUTPUT_DIR = MODEL_DIR / "eval_test_ckpt24"

# Patch hanya berlaku di proses evaluasi yang dijalankan di bawah.
wrapper = r"""
import sys
import runpy
import tensorflow as tf
from tensorflow.python.ops import control_flow_ops

# Alias kompatibilitas sementara untuk decoder yang mengakses
# control_flow_ops.cond.
if not hasattr(control_flow_ops, "cond"):
    control_flow_ops.cond = tf.cond

script = sys.argv[1]
script_args = sys.argv[2:]
sys.argv = [script] + script_args

print("Temporary compatibility alias enabled:", 
      hasattr(control_flow_ops, "cond"), flush=True)

runpy.run_path(script, run_name="__main__")
"""

env = os.environ.copy()
extra_paths = [
    str(RESEARCH_DIR),
    str(RESEARCH_DIR / "slim"),
]
if env.get("PYTHONPATH"):
    extra_paths.append(env["PYTHONPATH"])
env["PYTHONPATH"] = os.pathsep.join(extra_paths)

cmd = [
    sys.executable,
    "-c",
    wrapper,
    str(SCRIPT),
    f"--pipeline_config_path={TEST_CONFIG}",
    f"--model_dir={OUTPUT_DIR}",
    f"--checkpoint_dir={MODEL_DIR}",
    "--eval_timeout=120",
    "--alsologtostderr",
]

print("=== EVALUASI SSD PADA TEST ===")
print("Checkpoint:", MODEL_DIR / "ckpt-24")
print("Konfigurasi:", TEST_CONFIG)
print("Dataset test:", PROJECT_DIR / "dataset/tfrecord/test.record")

result = subprocess.run(
    cmd,
    cwd=str(RESEARCH_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=1800
)

print("\n=== OUTPUT EVALUASI ===")
print(result.stdout[-16000:])
print("\nReturn code:", result.returncode)

=== EVALUASI SSD PADA TEST ===
Checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-24
Konfigurasi: /home/rzangef/SSD-PPE/models/pipeline_ppe_test.config
Dataset test: /home/rzangef/SSD-PPE/dataset/tfrecord/test.record

=== OUTPUT EVALUASI ===
2026-10-20 07:12:14.583515: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 07:12:15.325936: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-20 07:12:15.326366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-20 07:12:15.475104: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register fact

In [2]:
# Konversi ke TF Record
from pathlib import Path
import re
import xml.etree.ElementTree as ET
import tensorflow as tf
from PIL import Image

# =====================================================
# 1. KONFIGURASI PATH
# =====================================================
YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
XML_ROOT = Path(
    "/home/rzangef/spark-data/CHVG-Dataset-original/CHVG-Dataset"
)
OUTPUT_DIR = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split"
)
LABEL_MAP = OUTPUT_DIR / "label_map.pbtxt"

SPLITS = ["train", "val", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# Jangan menimpa hasil konversi sebelumnya
for split in SPLITS:
    record_path = OUTPUT_DIR / f"{split}.record"
    if record_path.exists():
        raise FileExistsError(
            f"{record_path} sudah ada. "
            "Hentikan proses agar file lama tidak tertimpa."
        )

# =====================================================
# 2. MEMBACA LABEL MAP
# =====================================================
label_text = LABEL_MAP.read_text(encoding="utf-8")
items = re.findall(
    r"item\s*\{(.*?)\}", label_text, flags=re.DOTALL
)

class_to_id = {}

for item in items:
    id_match = re.search(r"\bid\s*:\s*(\d+)", item)
    name_match = re.search(
        r"\bname\s*:\s*['\"]([^'\"]+)['\"]", item
    )

    if id_match and name_match:
        class_id = int(id_match.group(1))
        class_name = name_match.group(1).strip().lower()
        class_to_id[class_name] = class_id

expected_classes = {
    "person", "vest", "yellow", "white",
    "glass", "blue", "red", "head"
}

assert set(class_to_id) == expected_classes, (
    f"Label map tidak sesuai: {class_to_id}"
)
assert sorted(class_to_id.values()) == list(range(1, 9)), (
    f"ID kelas tidak berurutan: {class_to_id}"
)

print("Label map:", class_to_id)

# =====================================================
# 3. MEMBUAT INDEKS XML BERDASARKAN NAMA FILE
# =====================================================
xml_index = {}

for xml_path in XML_ROOT.rglob("*.xml"):
    key = xml_path.stem.casefold()
    xml_index.setdefault(key, []).append(xml_path)

ambiguous_xml = {
    key: paths for key, paths in xml_index.items()
    if len(paths) != 1
}
if ambiguous_xml:
    raise ValueError(
        f"Ada XML dengan nama dasar duplikat: "
        f"{list(ambiguous_xml)[:10]}"
    )

xml_index = {
    key: paths[0] for key, paths in xml_index.items()
}

# =====================================================
# 4. FUNGSI TFRECORD
# =====================================================
def bytes_feature(value):
    return tf.train.Feature(
        bytes_list=tf.train.BytesList(value=[value])
    )

def bytes_list_feature(values):
    return tf.train.Feature(
        bytes_list=tf.train.BytesList(value=values)
    )

def int64_feature(value):
    return tf.train.Feature(
        int64_list=tf.train.Int64List(value=[int(value)])
    )

def int64_list_feature(values):
    return tf.train.Feature(
        int64_list=tf.train.Int64List(
            value=[int(v) for v in values]
        )
    )

def float_list_feature(values):
    return tf.train.Feature(
        float_list=tf.train.FloatList(
            value=[float(v) for v in values]
        )
    )

# =====================================================
# 5. KONVERSI SATU GAMBAR + XML
# =====================================================
def create_tf_example(image_path, xml_path):
    encoded = image_path.read_bytes()

    # Ambil dimensi aktual gambar
    with Image.open(image_path) as im:
        width, height = im.size
        image_format = im.format.lower()

    if width <= 0 or height <= 0:
        raise ValueError(f"Dimensi gambar tidak valid: {image_path}")

    if image_format == "jpg":
        image_format = "jpeg"

    root = ET.parse(xml_path).getroot()

    # Jika XML menyimpan dimensi, cocokkan dengan gambar aktual
    size_node = root.find("size")
    if size_node is not None:
        xml_width = size_node.findtext("width")
        xml_height = size_node.findtext("height")

        if xml_width and xml_height:
            if (
                int(float(xml_width)) != width
                or int(float(xml_height)) != height
            ):
                raise ValueError(
                    f"Dimensi XML berbeda dari gambar: {image_path.name}"
                )

    xmins, xmaxs, ymins, ymaxs = [], [], [], []
    class_texts, class_labels = [], []
    clipped_boxes = 0

    for obj in root.findall("object"):
        name = (obj.findtext("name") or "").strip().lower()

        if name not in class_to_id:
            raise ValueError(
                f"Kelas tidak dikenal '{name}' pada {xml_path.name}"
            )

        bbox = obj.find("bndbox")
        if bbox is None:
            raise ValueError(
                f"Bounding box tidak ditemukan: {xml_path.name}"
            )

        xmin = float(bbox.findtext("xmin"))
        ymin = float(bbox.findtext("ymin"))
        xmax = float(bbox.findtext("xmax"))
        ymax = float(bbox.findtext("ymax"))

        if xmax <= xmin or ymax <= ymin:
            raise ValueError(
                f"Bounding box tidak valid pada {xml_path.name}: "
                f"{xmin, ymin, xmax, ymax}"
            )

        # Batasi koordinat agar tetap berada di dalam gambar.
        # Jumlah box yang perlu dibatasi akan dilaporkan.
        old_box = (xmin, ymin, xmax, ymax)
        xmin = max(0.0, min(xmin, float(width)))
        xmax = max(0.0, min(xmax, float(width)))
        ymin = max(0.0, min(ymin, float(height)))
        ymax = max(0.0, min(ymax, float(height)))

        if old_box != (xmin, ymin, xmax, ymax):
            clipped_boxes += 1

        if xmax <= xmin or ymax <= ymin:
            raise ValueError(
                f"Bounding box berada di luar gambar: {xml_path.name}"
            )

        # Normalisasi koordinat ke rentang 0-1
        xmins.append(xmin / width)
        xmaxs.append(xmax / width)
        ymins.append(ymin / height)
        ymaxs.append(ymax / height)

        class_texts.append(name.encode("utf-8"))
        class_labels.append(class_to_id[name])

    feature_dict = {
        "image/height": int64_feature(height),
        "image/width": int64_feature(width),
        "image/filename": bytes_feature(
            image_path.name.encode("utf-8")
        ),
        "image/source_id": bytes_feature(
            image_path.stem.encode("utf-8")
        ),
        "image/encoded": bytes_feature(encoded),
        "image/format": bytes_feature(
            image_format.encode("utf-8")
        ),
        "image/object/bbox/xmin": float_list_feature(xmins),
        "image/object/bbox/xmax": float_list_feature(xmaxs),
        "image/object/bbox/ymin": float_list_feature(ymins),
        "image/object/bbox/ymax": float_list_feature(ymaxs),
        "image/object/class/text": bytes_list_feature(class_texts),
        "image/object/class/label": int64_list_feature(class_labels),
        "image/object/difficult": int64_list_feature(
            [0] * len(class_labels)
        ),
        "image/object/truncated": int64_list_feature(
            [0] * len(class_labels)
        ),
    }

    example = tf.train.Example(
        features=tf.train.Features(feature=feature_dict)
    )
    return example, len(class_labels), clipped_boxes

# =====================================================
# 6. KONVERSI TRAIN, VAL, TEST
# =====================================================
summary = {}
total_clipped = 0

for split in SPLITS:
    image_dir = YOLO_ROOT / "images" / split
    images = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not images:
        raise ValueError(f"Tidak ada gambar pada {image_dir}")

    temp_path = OUTPUT_DIR / f"{split}.record.tmp"
    final_path = OUTPUT_DIR / f"{split}.record"

    count_objects = 0
    count_empty = 0
    split_clipped = 0

    try:
        with tf.io.TFRecordWriter(str(temp_path)) as writer:
            for i, image_path in enumerate(images, start=1):
                key = image_path.stem.casefold()
                xml_path = xml_index.get(key)

                if xml_path is None:
                    raise FileNotFoundError(
                        f"XML tidak ditemukan untuk {image_path.name}"
                    )

                example, n_objects, n_clipped = create_tf_example(
                    image_path, xml_path
                )
                writer.write(example.SerializeToString())

                count_objects += n_objects
                count_empty += int(n_objects == 0)
                split_clipped += n_clipped

                if i % 200 == 0 or i == len(images):
                    print(f"{split}: {i}/{len(images)} gambar")

        # Ubah menjadi nama final hanya setelah split berhasil ditulis
        temp_path.replace(final_path)

    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise

    summary[split] = {
        "images": len(images),
        "objects": count_objects,
        "empty_images": count_empty,
        "clipped_boxes": split_clipped,
        "record": str(final_path),
        "size_MB": round(final_path.stat().st_size / (1024**2), 2),
    }
    total_clipped += split_clipped

    print(f"\nSelesai membuat {split}.record")
    print(summary[split])

print("\n=== RINGKASAN TFRECORD BARU ===")
for split, info in summary.items():
    print(f"\n{split.upper()}")
    for key, value in info.items():
        print(f"{key}: {value}")

print("\nTotal bounding box yang dibatasi:", total_clipped)
print("\nOutput:", OUTPUT_DIR)

2026-10-20 11:41:48.381692: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 11:41:48.609292: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-20 11:41:48.609482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-20 11:41:48.639559: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-20 11:41:48.743986: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 11:41:48.747737: I tensorflow/core/platform/cpu_feature_guard.cc:1

Label map: {'person': 1, 'vest': 2, 'yellow': 3, 'white': 4, 'glass': 5, 'blue': 6, 'red': 7, 'head': 8}
train: 200/1189 gambar
train: 400/1189 gambar
train: 600/1189 gambar
train: 800/1189 gambar
train: 1000/1189 gambar
train: 1189/1189 gambar

Selesai membuat train.record
{'images': 1189, 'objects': 8008, 'empty_images': 1, 'clipped_boxes': 24, 'record': '/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record', 'size_MB': 69.75}
val: 170/170 gambar

Selesai membuat val.record
{'images': 170, 'objects': 1211, 'empty_images': 0, 'clipped_boxes': 1, 'record': '/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record', 'size_MB': 9.78}
test: 200/340 gambar
test: 340/340 gambar

Selesai membuat test.record
{'images': 340, 'objects': 2385, 'empty_images': 0, 'clipped_boxes': 3, 'record': '/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record', 'size_MB': 19.99}

=== RINGKASAN TFRECORD BARU ===

TRAIN
images: 1189
objects: 8008
empty_images: 1
clipped_boxes: 24
record: /

In [3]:
# Validasi TF Record terhadap YOLO
from pathlib import Path
import tensorflow as tf
import numpy as np
from collections import Counter

YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
RECORD_ROOT = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split"
)

SPLITS = ["train", "val", "test"]
TOLERANCE = 0.003

def parse_yolo_labels(label_path):
    if not label_path.exists():
        return []

    boxes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue

        values = line.split()
        if len(values) != 5:
            raise ValueError(
                f"Format label tidak valid: {label_path}: {line}"
            )

        cls = int(values[0])
        coords = [float(v) for v in values[1:]]
        boxes.append((cls, *coords))

    return boxes

def parse_record_example(serialized):
    ex = tf.train.Example()
    ex.ParseFromString(serialized)
    f = ex.features.feature

    filename = f["image/filename"].bytes_list.value[0].decode(
        "utf-8"
    )

    class_ids = list(
        f["image/object/class/label"].int64_list.value
    )
    xmins = list(f["image/object/bbox/xmin"].float_list.value)
    xmaxs = list(f["image/object/bbox/xmax"].float_list.value)
    ymins = list(f["image/object/bbox/ymin"].float_list.value)
    ymaxs = list(f["image/object/bbox/ymax"].float_list.value)

    if not (
        len(class_ids) == len(xmins) == len(xmaxs)
        == len(ymins) == len(ymaxs)
    ):
        raise ValueError(
            f"Jumlah kelas dan koordinat tidak sama: {filename}"
        )

    # Ubah bbox SSD [xmin, ymin, xmax, ymax]
    # menjadi format YOLO [class, xc, yc, w, h].
    boxes = []
    for cls, xmin, ymin, xmax, ymax in zip(
        class_ids, xmins, ymins, xmaxs, ymaxs
    ):
        xc = (xmin + xmax) / 2
        yc = (ymin + ymax) / 2
        w = xmax - xmin
        h = ymax - ymin

        # ID kelas SSD mulai 1, YOLO mulai 0
        boxes.append((int(cls) - 1, xc, yc, w, h))

    return filename, boxes

def compare_boxes(record_boxes, yolo_boxes):
    """Cocokkan objek berdasarkan kelas dan koordinat."""
    if len(record_boxes) != len(yolo_boxes):
        return False

    remaining = list(yolo_boxes)

    for rec_box in record_boxes:
        rec_cls = rec_box[0]
        rec_coords = np.array(rec_box[1:], dtype=float)

        candidates = []
        for i, yolo_box in enumerate(remaining):
            if yolo_box[0] != rec_cls:
                continue

            yolo_coords = np.array(yolo_box[1:], dtype=float)
            diff = np.max(np.abs(rec_coords - yolo_coords))

            if diff <= TOLERANCE:
                candidates.append((diff, i))

        if not candidates:
            return False

        _, best_index = min(candidates)
        remaining.pop(best_index)

    return len(remaining) == 0

# =====================================================
# JALANKAN VALIDASI SETIAP SPLIT
# =====================================================
for split in SPLITS:
    record_path = RECORD_ROOT / f"{split}.record"
    label_dir = YOLO_ROOT / "labels" / split

    label_index = {}
    for path in label_dir.glob("*.txt"):
        key = path.stem.casefold()
        if key in label_index:
            raise ValueError(
                f"Label duplikat untuk nama gambar: {key}"
            )
        label_index[key] = path

    record_count = 0
    record_objects = 0
    yolo_objects = 0
    mismatch_images = []
    missing_label_files = []
    empty_record_images = 0
    class_record = Counter()
    class_yolo = Counter()

    seen_names = set()

    for serialized in tf.data.TFRecordDataset([str(record_path)]):
        filename, record_boxes = parse_record_example(
            serialized.numpy()
        )
        key = Path(filename).stem.casefold()

        if key in seen_names:
            raise ValueError(
                f"Nama gambar duplikat di TFRecord {split}: {filename}"
            )
        seen_names.add(key)

        record_count += 1
        record_objects += len(record_boxes)
        empty_record_images += int(len(record_boxes) == 0)

        for box in record_boxes:
            class_record[box[0]] += 1

        label_path = label_index.get(key)

        if label_path is None:
            missing_label_files.append(filename)
            yolo_boxes = []
        else:
            yolo_boxes = parse_yolo_labels(label_path)

        yolo_objects += len(yolo_boxes)
        for box in yolo_boxes:
            class_yolo[box[0]] += 1

        if not compare_boxes(record_boxes, yolo_boxes):
            mismatch_images.append({
                "filename": filename,
                "objects_tfrecord": len(record_boxes),
                "objects_yolo": len(yolo_boxes),
                "label_file_exists": label_path is not None,
            })

    print(f"\n{'=' * 15} {split.upper()} {'=' * 15}")
    print("Jumlah gambar TFRecord :", record_count)
    print("Jumlah objek TFRecord  :", record_objects)
    print("Jumlah objek label YOLO:", yolo_objects)
    print("Gambar tanpa objek      :", empty_record_images)
    print("File label YOLO tidak ditemukan:", len(missing_label_files))
    print("Gambar dengan anotasi berbeda   :", len(mismatch_images))
    print("Jumlah objek per kelas TFRecord :", dict(sorted(class_record.items())))
    print("Jumlah objek per kelas YOLO     :", dict(sorted(class_yolo.items())))

    if missing_label_files:
        print("Contoh label tidak ditemukan:", missing_label_files[:10])

    if mismatch_images:
        print("Contoh gambar yang berbeda:")
        for item in mismatch_images[:10]:
            print(item)


=============== TRAIN ===============
Jumlah gambar TFRecord : 1189
Jumlah objek TFRecord  : 8008
Jumlah objek label YOLO: 8008
Gambar tanpa objek      : 1
File label YOLO tidak ditemukan: 0
Gambar dengan anotasi berbeda   : 0
Jumlah objek per kelas TFRecord : {0: 3220, 1: 1474, 2: 841, 3: 899, 4: 380, 5: 343, 6: 364, 7: 487}
Jumlah objek per kelas YOLO     : {0: 3220, 1: 1474, 2: 841, 3: 899, 4: 380, 5: 343, 6: 364, 7: 487}

=============== VAL ===============
Jumlah gambar TFRecord : 170
Jumlah objek TFRecord  : 1211
Jumlah objek label YOLO: 1211
Gambar tanpa objek      : 0
File label YOLO tidak ditemukan: 0
Gambar dengan anotasi berbeda   : 0
Jumlah objek per kelas TFRecord : {0: 490, 1: 233, 2: 112, 3: 130, 4: 49, 5: 55, 6: 50, 7: 92}
Jumlah objek per kelas YOLO     : {0: 490, 1: 233, 2: 112, 3: 130, 4: 49, 5: 55, 6: 50, 7: 92}

=============== TEST ===============
Jumlah gambar TFRecord : 340
Jumlah objek TFRecord  : 2385
Jumlah objek label YOLO: 2385
Gambar tanpa objek      : 0


In [4]:
# pipeline_ppe_yolo_split.config untuk training SSD dengan data train dan evaluasi validasi
# pipeline_ppe_yolo_split_test.config untuk evaluasi akhir menggunakan data test yang sama dengan YOLO

# Konfigurasi SSD Baru

from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")
MODELS_DIR = PROJECT / "models"
DATA_DIR = PROJECT / "dataset/tfrecord_yolo_split"

OLD_CONFIG = MODELS_DIR / "pipeline_ppe.config"

TRAIN_CONFIG = MODELS_DIR / "pipeline_ppe_yolo_split.config"
TEST_CONFIG = MODELS_DIR / "pipeline_ppe_yolo_split_test.config"

COCO_CKPT = (
    MODELS_DIR
    / "ssd_mobilenet_v2_fpnlite_320x320"
    / "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
    / "checkpoint/ckpt-0"
)

# Pastikan seluruh file input tersedia
required_files = [
    OLD_CONFIG,
    DATA_DIR / "train.record",
    DATA_DIR / "val.record",
    DATA_DIR / "test.record",
    DATA_DIR / "label_map.pbtxt",
    Path(str(COCO_CKPT) + ".index"),
]

for path in required_files:
    print(f"{path}: {path.exists()}")
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

# Jangan menimpa konfigurasi hasil eksperimen sebelumnya
if TRAIN_CONFIG.exists() or TEST_CONFIG.exists():
    raise FileExistsError(
        "Konfigurasi baru sudah ada. Periksa file terlebih dahulu "
        "agar tidak tertimpa."
    )

config_text = OLD_CONFIG.read_text(encoding="utf-8")

# Ganti lokasi dataset lama dengan dataset split YOLO
replacements = {
    "/home/rzangef/SSD-PPE/dataset/tfrecord/train.record":
        str(DATA_DIR / "train.record"),

    "/home/rzangef/SSD-PPE/dataset/tfrecord/val.record":
        str(DATA_DIR / "val.record"),

    "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt":
        str(DATA_DIR / "label_map.pbtxt"),
}

for old_path, new_path in replacements.items():
    occurrences = config_text.count(old_path)
    if occurrences == 0:
        raise ValueError(
            f"Path lama tidak ditemukan di konfigurasi: {old_path}"
        )
    config_text = config_text.replace(old_path, new_path)
    print(f"\nDiganti ({occurrences} kali):")
    print("Lama :", old_path)
    print("Baru :", new_path)

# Buat konfigurasi training dengan train.record dan val.record
TRAIN_CONFIG.write_text(config_text, encoding="utf-8")

# Buat konfigurasi evaluasi test berdasarkan konfigurasi baru.
# Hanya ganti path evaluasi val.record menjadi test.record.
test_text = config_text.replace(
    str(DATA_DIR / "val.record"),
    str(DATA_DIR / "test.record"),
    1
)

if test_text == config_text:
    raise ValueError("Path val.record tidak berhasil diganti untuk test.")

TEST_CONFIG.write_text(test_text, encoding="utf-8")

print("\n=== KONFIGURASI BARU DIBUAT ===")
print("Training:", TRAIN_CONFIG)
print("Test    :", TEST_CONFIG)

/home/rzangef/SSD-PPE/models/pipeline_ppe.config: True
/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record: True
/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record: True
/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record: True
/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt: True
/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0.index: True

Diganti (1 kali):
Lama : /home/rzangef/SSD-PPE/dataset/tfrecord/train.record
Baru : /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record

Diganti (1 kali):
Lama : /home/rzangef/SSD-PPE/dataset/tfrecord/val.record
Baru : /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record

Diganti (2 kali):
Lama : /home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt
Baru : /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt

=== KONFIGURASI BARU DIBUAT ===
Training: /home/rzangef/SSD-PPE/models/pipel

In [5]:
# Verifikasi sebelum training
import re

def inspect_config(path):
    text = path.read_text(encoding="utf-8")

    print(f"\n=== {path.name} ===")

    for pattern in [
        r"num_classes\s*:\s*\d+",
        r"batch_size\s*:\s*\d+",
        r"num_steps\s*:\s*\d+",
        r"fine_tune_checkpoint\s*:\s*['\"][^'\"]+['\"]",
        r"fine_tune_checkpoint_type\s*:\s*['\"][^'\"]+['\"]",
        r"input_path\s*:\s*['\"][^'\"]+['\"]",
        r"label_map_path\s*:\s*['\"][^'\"]+['\"]",
    ]:
        for match in re.findall(pattern, text):
            print(match)

    print("Jumlah kelas 8:", bool(
        re.search(r"num_classes\s*:\s*8\b", text)
    ))
    print("Mengarah ke dataset baru:", str(DATA_DIR) in text)

inspect_config(TRAIN_CONFIG)
inspect_config(TEST_CONFIG)


=== pipeline_ppe_yolo_split.config ===
num_classes: 8
batch_size: 1
num_steps: 50000
fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
fine_tune_checkpoint_type: "detection"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"
label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
Jumlah kelas 8: True
Mengarah ke dataset baru: True

=== pipeline_ppe_yolo_split_test.config ===
num_classes: 8
batch_size: 1
num_steps: 50000
fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
fine_tune_checkpoint_type: "detection"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.reco

In [6]:
# Hitung otomatis num steps agar sama seperti epoch 5, 10, 15
import math
jumlah_gambar = 1189
batch_size = 1

steps_per_epoch = math.ceil(jumlah_gambar / batch_size)

print("Jumlah gambar training:", jumlah_gambar)
print("Batch size:", batch_size)
print("Perkiraan steps per epoch:", steps_per_epoch)

for epoch in [5, 10, 15]:
    num_steps = epoch * steps_per_epoch
    print(f"{epoch} epoch ekuivalen = {num_steps} steps")

Jumlah gambar training: 1189
Batch size: 1
Perkiraan steps per epoch: 1189
5 epoch ekuivalen = 5945 steps
10 epoch ekuivalen = 11890 steps
15 epoch ekuivalen = 17835 steps


In [7]:
# Periksan SSD

from pathlib import Path
import re
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"

CONFIG = PROJECT / "models/pipeline_ppe_yolo_split.config"
DATA_DIR = PROJECT / "dataset/tfrecord_yolo_split"

MODEL_DIR = (
    PROJECT
    / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

COCO_CKPT = (
    PROJECT
    / "models/ssd_mobilenet_v2_fpnlite_320x320/"
      "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
)

print("=== 1. FILE PENTING ===")

files = {
    "Python environment SSD": PYTHON,
    "Pipeline config": CONFIG,
    "Train TFRecord": DATA_DIR / "train.record",
    "Validation TFRecord": DATA_DIR / "val.record",
    "Test TFRecord": DATA_DIR / "test.record",
    "Label map": DATA_DIR / "label_map.pbtxt",
    "Checkpoint COCO": Path(str(COCO_CKPT) + ".index"),
}

all_ok = True

for name, path in files.items():
    ok = path.exists()
    print(f"[{'OK' if ok else 'BELUM ADA'}] {name}")
    if not ok:
        print("    ", path)
        all_ok = False

print("\n=== 2. ENVIRONMENT ===")

if PYTHON.exists():
    result = subprocess.run(
        [
            str(PYTHON), "-c",
            "import sys, tensorflow as tf; "
            "print('Python:', sys.executable); "
            "print('TensorFlow:', tf.__version__)"
        ],
        capture_output=True,
        text=True,
        timeout=60
    )
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr[-2000:])
        all_ok = False

print("=== 3. KONFIGURASI ===")

if CONFIG.exists():
    text_config = CONFIG.read_text()

    for pattern in [
        r"num_classes:\s*\d+",
        r"batch_size:\s*\d+",
        r"num_steps:\s*\d+",
        r"fine_tune_checkpoint:\s*['\"]?([^'\"]+)",
    ]:
        match = re.search(pattern, text_config)
        print(match.group(0) if match else f"Tidak ditemukan: {pattern}")

    print("\nReferensi dataset dalam config:")
    for line in text_config.splitlines():
        if "input_path:" in line or "label_map_path:" in line:
            print(line.strip())

print("\n=== 4. FOLDER OUTPUT ===")
print("Lokasi:", MODEL_DIR)

if MODEL_DIR.exists():
    existing = list(MODEL_DIR.iterdir())
    print("Jumlah item:", len(existing))
    for p in existing[:15]:
        print("-", p.name)

    if any(MODEL_DIR.glob("ckpt-*.index")):
        print("PERHATIAN: checkpoint sudah ada; periksa sebelum melanjutkan.")
else:
    print("Folder belum ada; belum ada checkpoint di lokasi ini.")

print("\nSTATUS PEMERIKSAAN FILE:", "OK" if all_ok else "ADA YANG PERLU DIPERIKSA")


=== 1. FILE PENTING ===
[OK] Python environment SSD
[OK] Pipeline config
[OK] Train TFRecord
[OK] Validation TFRecord
[OK] Test TFRecord
[OK] Label map
[OK] Checkpoint COCO

=== 2. ENVIRONMENT ===
Python: /home/rzangef/SSD-PPE/.venv/bin/python
TensorFlow: 2.15.1

=== 3. KONFIGURASI ===
num_classes: 8
batch_size: 1
num_steps: 50000
fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0

Referensi dataset dalam config:
label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"
label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"

=== 4. FOLDER OUTPUT ===
Lokasi: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split
Folder belum ada; belum ada checkpoint di lokasi ini.

S

In [8]:

from pathlib import Path
import re

PROJECT = Path("/home/rzangef/SSD-PPE")
RESEARCH = PROJECT / "object-detection-api/research"
SCRIPT = RESEARCH / "object_detection/model_main_tf2.py"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split.config"
DATA_DIR = PROJECT / "dataset/tfrecord_yolo_split"
MODEL_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"

print("=== A. SCRIPT TRAINING ===")
print("Script tersedia:", SCRIPT.exists())
print("Lokasi:", SCRIPT)

print("\n=== B. KONFIGURASI TRAINING ===")
config_text = CONFIG.read_text()

# Tampilkan blok train_config untuk memeriksa num_steps
match = re.search(
    r"train_config\s*\{.*?\n\}",
    config_text,
    flags=re.DOTALL
)

if match:
    print(match.group(0)[:6000])
else:
    print("Blok train_config tidak ditemukan; perlu pemeriksaan manual.")

print("\n=== C. INPUT DATASET ===")
for filename in ["train.record", "val.record", "test.record"]:
    path = DATA_DIR / filename
    print(
        f"{filename}:",
        "ADA" if path.exists() else "TIDAK ADA",
        f"({path.stat().st_size / (1024**2):.2f} MB)"
        if path.exists() else ""
    )

print("\n=== D. FOLDER OUTPUT ===")
print("Folder output sudah ada:", MODEL_DIR.exists())
if MODEL_DIR.exists():
    checkpoints = list(MODEL_DIR.glob("ckpt-*.index"))
    print("Jumlah checkpoint:", len(checkpoints))
    for ckpt in checkpoints:
        print("-", ckpt.name)

print("\n=== E. TARGET TAHAP PERTAMA ===")
jumlah_gambar = 1189
batch_size = 1
target_steps = 5 * ((jumlah_gambar + batch_size - 1) // batch_size)

print("Target steps:", target_steps)
print("Target epoch ekuivalen:", 5)


=== A. SCRIPT TRAINING ===
Script tersedia: True
Lokasi: /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_main_tf2.py

=== B. KONFIGURASI TRAINING ===
train_config {
  batch_size: 1
  data_augmentation_options {
    random_horizontal_flip {
    }
  }
  data_augmentation_options {
    random_crop_image {
      min_object_covered: 0.0
      min_aspect_ratio: 0.75
      max_aspect_ratio: 3.0
      min_area: 0.75
      max_area: 1.0
      overlap_thresh: 0.0
    }
  }
  sync_replicas: true
  optimizer {
    momentum_optimizer {
      learning_rate {
        cosine_decay_learning_rate {
          learning_rate_base: 0.07999999821186066
          total_steps: 50000
          warmup_learning_rate: 0.026666000485420227
          warmup_steps: 1000
        }
      }
      momentum_optimizer_value: 0.8999999761581421
    }
    use_moving_average: false
  }
  fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320

In [9]:
# Konfigurasi khusus num step 5945 = 5 epoch
# pipeline_ppe_yolo_split_stage5.config
from pathlib import Path
import re

PROJECT = Path("/home/rzangef/SSD-PPE")
SOURCE_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split.config"
STAGE5_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"

text_config = SOURCE_CONFIG.read_text()

# Ambil blok train_config utama saja
match = re.search(
    r"(?ms)^train_config\s*\{.*?^\}",
    text_config
)

if not match:
    raise RuntimeError("Blok train_config tidak ditemukan. File tidak diubah.")

block = match.group(0)

# Sesuaikan target jumlah langkah
block, n_steps = re.subn(
    r"(?m)^(\s*num_steps:\s*)\d+",
    r"\g<1>5945",
    block
)

# Sesuaikan jadwal learning rate untuk batch size 1
# Nilai ini merupakan pengaturan awal untuk eksperimen fine-tuning.
block, n_total = re.subn(
    r"(?m)^(\s*total_steps:\s*)\d+",
    r"\g<1>5945",
    block
)
block, n_warmup = re.subn(
    r"(?m)^(\s*warmup_steps:\s*)\d+",
    r"\g<1>200",
    block
)
block, n_lr = re.subn(
    r"(?m)^(\s*learning_rate_base:\s*)[0-9.eE+-]+",
    r"\g<1>0.0003",
    block
)
block, n_warmup_lr = re.subn(
    r"(?m)^(\s*warmup_learning_rate:\s*)[0-9.eE+-]+",
    r"\g<1>0.0001",
    block
)

if not all([n_steps, n_total, n_warmup, n_lr, n_warmup_lr]):
    raise RuntimeError(
        "Ada parameter yang tidak berhasil diubah. "
        "Konfigurasi baru belum disimpan."
    )

new_config = (
    text_config[:match.start()]
    + block
    + text_config[match.end():]
)

# Pastikan file baru tidak menimpa file yang sudah ada
if STAGE5_CONFIG.exists():
    raise FileExistsError(
        f"File konfigurasi sudah ada: {STAGE5_CONFIG}. "
        "Jangan menimpanya sebelum diperiksa."
    )

STAGE5_CONFIG.write_text(new_config)

print("Konfigurasi tahap pertama berhasil dibuat:")
print(STAGE5_CONFIG)

print("\n=== PARAMETER YANG DIATUR ===")
for line in block.splitlines():
    if any(key in line for key in [
        "batch_size:",
        "num_steps:",
        "total_steps:",
        "warmup_steps:",
        "learning_rate_base:",
        "warmup_learning_rate:",
        "fine_tune_checkpoint:",
        "fine_tune_checkpoint_type:",
    ]):
        print(line.strip())

print("\n=== PEMERIKSAAN DATASET ===")
for line in new_config.splitlines():
    if "input_path:" in line or "label_map_path:" in line:
        print(line.strip())

print("\nKonfigurasi asli tetap:", SOURCE_CONFIG.exists())

Konfigurasi tahap pertama berhasil dibuat:
/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config

=== PARAMETER YANG DIATUR ===
batch_size: 1
learning_rate_base: 0.0003
total_steps: 5945
warmup_learning_rate: 0.0001
warmup_steps: 200
fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
num_steps: 5945
fine_tune_checkpoint_type: "detection"

=== PEMERIKSAAN DATASET ===
label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"
label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"

Konfigurasi asli tetap: True


In [10]:
from pathlib import Path
import os
import shutil
import subprocess
import psutil

# Lokasi proyek
PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH = PROJECT / "object-detection-api/research"

CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"
MODEL_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
TFRECORD_DIR = PROJECT / "dataset/tfrecord_yolo_split"
CHECKPOINT = PROJECT / (
    "models/ssd_mobilenet_v2_fpnlite_320x320/"
    "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
)

print("=== 1. PEMERIKSAAN FILE ===")
files = {
    "Python environment": PYTHON,
    "Training script": RESEARCH / "object_detection/model_main_tf2.py",
    "Config Stage 5": CONFIG,
    "Train TFRecord": TFRECORD_DIR / "train.record",
    "Validation TFRecord": TFRECORD_DIR / "val.record",
    "Test TFRecord": TFRECORD_DIR / "test.record",
    "Label map": TFRECORD_DIR / "label_map.pbtxt",
    "Checkpoint pretrained": Path(str(CHECKPOINT) + ".index"),
}

for name, path in files.items():
    print(f"[{'OK' if path.exists() else 'TIDAK ADA'}] {name}: {path}")

print("\n=== 2. KONDISI SISTEM ===")
print(f"CPU logical cores : {psutil.cpu_count(logical=True)}")
print(f"RAM total         : {psutil.virtual_memory().total / 1024**3:.2f} GB")
print(f"RAM tersedia      : {psutil.virtual_memory().available / 1024**3:.2f} GB")
print(f"RAM terpakai      : {psutil.virtual_memory().percent}%")
print(f"Disk kosong       : {shutil.disk_usage(PROJECT).free / 1024**3:.2f} GB")

print("\n=== 3. KONDISI OUTPUT ===")
print(f"Folder output     : {MODEL_DIR}")
print(f"Sudah ada         : {MODEL_DIR.exists()}")

if MODEL_DIR.exists():
    existing = list(MODEL_DIR.glob("ckpt-*.index"))
    print(f"Checkpoint lama   : {[p.name for p in existing]}")

print("\n=== 4. KONFIGURASI STAGE 5 ===")
if CONFIG.exists():
    config_text = CONFIG.read_text()
    for keyword in [
        "batch_size:",
        "num_steps:",
        "learning_rate_base:",
        "warmup_steps:",
        "fine_tune_checkpoint:",
        "num_classes:",
    ]:
        print(f"\n--- {keyword} ---")
        matches = [
            line.strip()
            for line in config_text.splitlines()
            if keyword in line
        ]
        print("\n".join(matches) if matches else "Tidak ditemukan")

=== 1. PEMERIKSAAN FILE ===
[OK] Python environment: /home/rzangef/SSD-PPE/.venv/bin/python
[OK] Training script: /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_main_tf2.py
[OK] Config Stage 5: /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config
[OK] Train TFRecord: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record
[OK] Validation TFRecord: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record
[OK] Test TFRecord: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record
[OK] Label map: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt
[OK] Checkpoint pretrained: /home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0.index

=== 2. KONDISI SISTEM ===
CPU logical cores : 4
RAM total         : 7.76 GB
RAM tersedia      : 6.61 GB
RAM terpakai      : 14.8%
Disk kosong       : 930.02 GB

=== 3. KONDISI OUTPUT ===
Folder output     : /home/rz

In [11]:
# Training 5945 steps
import os
import subprocess
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")
RESEARCH = PROJECT / "object-detection-api/research"
PYTHON = PROJECT / ".venv/bin/python"

CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"
SCRIPT = RESEARCH / "object_detection/model_main_tf2.py"
MODEL_DIR = PROJECT / (
    "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)
LOG_DIR = PROJECT / "logs"
LOG_FILE = LOG_DIR / "ssd_stage5_training.log"
WRAPPER = PROJECT / "run_ssd_stage5_wrapper.py"

# Pastikan file utama tersedia
for path in [PYTHON, SCRIPT, CONFIG]:
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

# Jangan menimpa hasil training yang sudah ada
if MODEL_DIR.exists() and any(MODEL_DIR.iterdir()):
    raise RuntimeError(
        f"Folder output sudah berisi file: {MODEL_DIR}\n"
        "Periksa checkpoint sebelum menjalankan ulang."
    )

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Kompatibilitas untuk environment TensorFlow Object Detection API
WRAPPER.write_text(r'''
import sys
import runpy
import tensorflow as tf
from tensorflow.python.ops import control_flow_ops

if not hasattr(control_flow_ops, "cond"):
    control_flow_ops.cond = tf.cond

print("Compatibility alias:", hasattr(control_flow_ops, "cond"), flush=True)

script = sys.argv[1]
args = sys.argv[2:]
sys.argv = [script] + args
runpy.run_path(script, run_name="__main__")
''')

env = os.environ.copy()
paths = [str(RESEARCH), str(RESEARCH / "slim")]
if env.get("PYTHONPATH"):
    paths.append(env["PYTHONPATH"])
env["PYTHONPATH"] = os.pathsep.join(paths)
env["TF_CPP_MIN_LOG_LEVEL"] = "2"

command = [
    str(PYTHON),
    str(WRAPPER),
    str(SCRIPT),
    f"--pipeline_config_path={CONFIG}",
    f"--model_dir={MODEL_DIR}",
    "--alsologtostderr",
]

# Jalankan training di latar belakang dan simpan log
with open(LOG_FILE, "w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(RESEARCH),
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Training SSD Stage 5 telah diluncurkan.")
print("PID       :", process.pid)
print("Target    : 5945 steps")
print("Config    :", CONFIG)
print("Output    :", MODEL_DIR)
print("Log       :", LOG_FILE)

Training SSD Stage 5 telah diluncurkan.
PID       : 4102
Target    : 5945 steps
Config    : /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config
Output    : /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split
Log       : /home/rzangef/SSD-PPE/logs/ssd_stage5_training.log


In [12]:
from pathlib import Path

log_file = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log"
)

if log_file.exists():
    lines = log_file.read_text(errors="replace").splitlines()
    print("\n".join(lines[-30:]))
else:
    print("Log belum tersedia.")

rename to distribute_datasets_from_function
W1020 14:59:27.517634 140488109375488 deprecation.py:50] From /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_lib_v2.py:563: StrategyBase.experimental_distribute_datasets_from_function (from tensorflow.python.distribute.distribute_lib) is deprecated and will be removed in a future version.
Instructions for updating:
rename to distribute_datasets_from_function
INFO:tensorflow:Reading unweighted datasets: ['/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record']
I1020 14:59:27.536221 140488109375488 dataset_builder.py:162] Reading unweighted datasets: ['/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record']
INFO:tensorflow:Reading record datasets for input file: ['/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record']
I1020 14:59:27.536777 140488109375488 dataset_builder.py:79] Reading record datasets for input file: ['/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record']
INFO:t

In [13]:
# Cek apakah proses masih berjalan
import psutil

pid = 4102

if psutil.pid_exists(pid):
    proc = psutil.Process(pid)
    print("Status proses :", proc.status())
    print("CPU proses    :", proc.cpu_percent(interval=1), "%")
    print("RAM proses    :", round(proc.memory_info().rss / 1024**2, 1), "MB")
else:
    print("Proses training sudah berhenti.")

Status proses : sleeping
CPU proses    : 235.2 %
RAM proses    : 1933.9 MB


In [14]:
# training menghasilkan loss?
from pathlib import Path

log_file = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log"
)

lines = log_file.read_text(errors="replace").splitlines()

print("\n".join(lines[-40:]))

print("\n=== PENCARIAN LOSS ===")
for line in lines:
    if "Loss/" in line or "loss =" in line.lower() or "step =" in line.lower():
        print(line)

 'Loss/localization_loss': 0.7106733,
 'Loss/regularization_loss': 0.15344411,
 'Loss/total_loss': 1.9939586,
 'learning_rate': 0.00020000001}
INFO:tensorflow:Step 200 per-step time 0.304s
I1020 15:01:55.514907 140488109375488 model_lib_v2.py:705] Step 200 per-step time 0.304s
INFO:tensorflow:{'Loss/classification_loss': 1.1309416,
 'Loss/localization_loss': 0.65493006,
 'Loss/regularization_loss': 0.15344116,
 'Loss/total_loss': 1.9393128,
 'learning_rate': 0.0003}
I1020 15:01:55.515335 140488109375488 model_lib_v2.py:708] {'Loss/classification_loss': 1.1309416,
 'Loss/localization_loss': 0.65493006,
 'Loss/regularization_loss': 0.15344116,
 'Loss/total_loss': 1.9393128,
 'learning_rate': 0.0003}
INFO:tensorflow:Step 300 per-step time 0.300s
I1020 15:02:25.538582 140488109375488 model_lib_v2.py:705] Step 300 per-step time 0.300s
INFO:tensorflow:{'Loss/classification_loss': 1.1255244,
 'Loss/localization_loss': 0.71377206,
 'Loss/regularization_loss': 0.1534375,
 'Loss/total_loss': 1.9

In [15]:
# Periksa proses training
import psutil

pid = 4102

if psutil.pid_exists(pid):
    p = psutil.Process(pid)
    print("PID          :", pid)
    print("Status       :", p.status())
    print("CPU          :", p.cpu_percent(interval=1), "%")
    print("RAM           :", round(p.memory_info().rss / 1024**2, 1), "MB")
    print("Mulai berjalan:", p.create_time())
else:
    print("Proses PID 4102 sudah tidak ditemukan.")

PID          : 4102
Status       : sleeping
CPU          : 241.7 %
RAM           : 1954.3 MB
Mulai berjalan: 1792483139.98


In [16]:
# Cek apakah bertambah
from pathlib import Path

log = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log"
)

lines = log.read_text(errors="replace").splitlines()

steps = [
    line.strip()
    for line in lines
    if "Step " in line and "per-step time" in line
]

print("Step terakhir:", steps[-1] if steps else "Belum ditemukan")

Step terakhir: I1020 15:10:31.319946 140488109375488 model_lib_v2.py:705] Step 1900 per-step time 0.308s


In [17]:
# Cek lagi
from pathlib import Path

log = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log"
)

lines = log.read_text(errors="replace").splitlines()

steps = [
    line.strip()
    for line in lines
    if "Step " in line and "per-step time" in line
]

print("Step terakhir:", steps[-1] if steps else "Belum ditemukan")

Step terakhir: I1020 15:21:57.148060 140488109375488 model_lib_v2.py:705] Step 4200 per-step time 0.293s


In [18]:
# Cek lagi
from pathlib import Path

log = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log"
)

lines = log.read_text(errors="replace").splitlines()

steps = [
    line.strip()
    for line in lines
    if "Step " in line and "per-step time" in line
]

print("Step terakhir:", steps[-1] if steps else "Belum ditemukan")

Step terakhir: I1020 15:27:29.052008 140488109375488 model_lib_v2.py:705] Step 5300 per-step time 0.300s


In [19]:
# Cek lagi
from pathlib import Path

log = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log"
)

lines = log.read_text(errors="replace").splitlines()

steps = [
    line.strip()
    for line in lines
    if "Step " in line and "per-step time" in line
]

print("Step terakhir:", steps[-1] if steps else "Belum ditemukan")

Step terakhir: I1020 15:31:04.956022 140488109375488 model_lib_v2.py:705] Step 6000 per-step time 0.296s


In [20]:
# Cek status
import subprocess

PID = 4102

result = subprocess.run(
    ["ps", "-p", str(PID), "-o", "pid,stat,%cpu,%mem,etime,cmd"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout.strip() else "Proses training sudah tidak berjalan.")

    PID STAT %CPU %MEM     ELAPSED CMD
   4102 Zs    206  0.0       38:08 [model_main_tf2.] <defunct>



In [21]:
from pathlib import Path

log_path = Path("/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log")
model_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

print("=== LOG TERAKHIR ===")
if log_path.exists():
    lines = log_path.read_text(errors="ignore").splitlines()
    for line in lines[-15:]:
        print(line)

print("\n=== CHECKPOINT ===")
for path in sorted(model_dir.glob("ckpt-*.index")):
    print(path.name)

=== LOG TERAKHIR ===
 'Loss/regularization_loss': 0.15335563,
 'Loss/total_loss': 0.8675599,
 'learning_rate': 4.5409802e-08}
INFO:tensorflow:Step 6000 per-step time 0.296s
I1020 15:31:04.956022 140488109375488 model_lib_v2.py:705] Step 6000 per-step time 0.296s
INFO:tensorflow:{'Loss/classification_loss': 0.75919217,
 'Loss/localization_loss': 0.53815883,
 'Loss/regularization_loss': 0.15335563,
 'Loss/total_loss': 1.4507067,
 'learning_rate': 0.0}
I1020 15:31:04.956441 140488109375488 model_lib_v2.py:708] {'Loss/classification_loss': 0.75919217,
 'Loss/localization_loss': 0.53815883,
 'Loss/regularization_loss': 0.15335563,
 'Loss/total_loss': 1.4507067,
 'learning_rate': 0.0}

=== CHECKPOINT ===
ckpt-1.index
ckpt-2.index
ckpt-3.index
ckpt-4.index
ckpt-5.index
ckpt-6.index
ckpt-7.index


In [22]:
# Cek isi checkpoint
from pathlib import Path

model_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

checkpoint_state = model_dir / "checkpoint"

print("=== CHECKPOINT STATE ===")
if checkpoint_state.exists():
    print(checkpoint_state.read_text())
else:
    print("File checkpoint tidak ditemukan.")

print("\n=== DETAIL CHECKPOINT ===")
for p in sorted(model_dir.glob("ckpt-*")):
    if p.is_file():
        print(f"{p.name:35} {p.stat().st_size / (1024**2):.2f} MB")

=== CHECKPOINT STATE ===
model_checkpoint_path: "ckpt-7"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_paths: "ckpt-2"
all_model_checkpoint_paths: "ckpt-3"
all_model_checkpoint_paths: "ckpt-4"
all_model_checkpoint_paths: "ckpt-5"
all_model_checkpoint_paths: "ckpt-6"
all_model_checkpoint_paths: "ckpt-7"
all_model_checkpoint_timestamps: 1792483216.0080063
all_model_checkpoint_timestamps: 1792483558.2438378
all_model_checkpoint_timestamps: 1792483861.868789
all_model_checkpoint_timestamps: 1792484159.868539
all_model_checkpoint_timestamps: 1792484458.692329
all_model_checkpoint_timestamps: 1792484758.62851
all_model_checkpoint_timestamps: 1792485066.0702882
last_preserved_timestamp: 1792483214.359389


=== DETAIL CHECKPOINT ===
ckpt-1.data-00000-of-00001          10.01 MB
ckpt-1.index                        0.03 MB
ckpt-2.data-00000-of-00001          19.85 MB
ckpt-2.index                        0.05 MB
ckpt-3.data-00000-of-00001          19.85 MB
ckpt-3.index                  

In [23]:
# Cek lagi
from pathlib import Path

log = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_training.log"
)

lines = log.read_text(errors="replace").splitlines()

steps = [
    line.strip()
    for line in lines
    if "Step " in line and "per-step time" in line
]

print("Step terakhir:", steps[-1] if steps else "Belum ditemukan")

Step terakhir: I1020 15:31:04.956022 140488109375488 model_lib_v2.py:705] Step 6000 per-step time 0.296s


In [24]:
# Periksa step yang tersimpan pada checkpoint
from pathlib import Path
import tensorflow as tf

model_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

print("=== STEP DALAM CHECKPOINT ===")

for index_file in sorted(model_dir.glob("ckpt-*.index")):
    checkpoint_prefix = str(index_file)[:-len(".index")]

    try:
        reader = tf.train.load_checkpoint(checkpoint_prefix)
        variables = reader.get_variable_to_shape_map()

        step_candidates = [
            name for name in variables
            if name in ("global_step", "global_step/.ATTRIBUTES/VARIABLE_VALUE")
        ]

        if step_candidates:
            for name in step_candidates:
                print(f"{index_file.stem}: {reader.get_tensor(name)} steps")
        else:
            print(f"{index_file.stem}: variabel global_step tidak ditemukan")

    except Exception as e:
        print(f"{index_file.stem}: gagal dibaca: {e}")

=== STEP DALAM CHECKPOINT ===
ckpt-1: variabel global_step tidak ditemukan
ckpt-2: variabel global_step tidak ditemukan
ckpt-3: variabel global_step tidak ditemukan
ckpt-4: variabel global_step tidak ditemukan
ckpt-5: variabel global_step tidak ditemukan
ckpt-6: variabel global_step tidak ditemukan
ckpt-7: variabel global_step tidak ditemukan


In [25]:
# Tampilkan variabel yang tersimpan di ckpt-7
import tensorflow as tf
from pathlib import Path

checkpoint_path = (
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"
)

reader = tf.train.load_checkpoint(checkpoint_path)
variables = reader.get_variable_to_shape_map()

print("Jumlah variabel:", len(variables))
print("\n=== VARIABEL YANG BERKAITAN DENGAN STEP ===")

found = False

for name, shape in variables.items():
    if "step" in name.lower() or "save_counter" in name.lower():
        print(f"{name} | shape={shape}")
        if "step" in name.lower():
            found = True

if not found:
    print("Tidak ditemukan nama variabel yang mengandung 'step'.")

print("\n=== 30 NAMA VARIABEL PERTAMA ===")
for name in list(variables.keys())[:30]:
    print(name, "|", variables[name])

Jumlah variabel: 622

=== VARIABEL YANG BERKAITAN DENGAN STEP ===
step/.ATTRIBUTES/VARIABLE_VALUE | shape=[]
save_counter/.ATTRIBUTES/VARIABLE_VALUE | shape=[]

=== 30 NAMA VARIABEL PERTAMA ===
step/.ATTRIBUTES/VARIABLE_VALUE | []
optimizer/decay/.ATTRIBUTES/VARIABLE_VALUE | []
model/_feature_extractor/classification_backbone/layer_with_weights-99/moving_variance/.ATTRIBUTES/VARIABLE_VALUE | [960]
model/_feature_extractor/classification_backbone/layer_with_weights-99/gamma/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE | [960]
model/_feature_extractor/classification_backbone/layer_with_weights-99/gamma/.ATTRIBUTES/VARIABLE_VALUE | [960]
model/_feature_extractor/classification_backbone/layer_with_weights-99/beta/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE | [960]
model/_feature_extractor/classification_backbone/layer_with_weights-97/moving_variance/.ATTRIBUTES/VARIABLE_VALUE | [960]
model/_feature_extractor/classification_backbone/layer_with_weights-97/m

In [26]:
# Baca nilai step semua checkpoint
import tensorflow as tf
from pathlib import Path

model_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

print("=== NILAI STEP CHECKPOINT SSD ===")

for index_file in sorted(
    model_dir.glob("ckpt-*.index"),
    key=lambda p: int(p.stem.split("-")[1])
):
    checkpoint_path = str(index_file)[:-len(".index")]

    try:
        reader = tf.train.load_checkpoint(checkpoint_path)
        variables = reader.get_variable_to_shape_map()

        step_name = "step/.ATTRIBUTES/VARIABLE_VALUE"

        if step_name in variables:
            step_value = reader.get_tensor(step_name)
            print(f"{index_file.stem}: step = {int(step_value)}")
        else:
            print(f"{index_file.stem}: variabel step tidak ditemukan")

    except Exception as e:
        print(f"{index_file.stem}: ERROR - {e}")

=== NILAI STEP CHECKPOINT SSD ===
ckpt-1: step = 0
ckpt-2: step = 1000
ckpt-3: step = 2000
ckpt-4: step = 3000
ckpt-5: step = 4000
ckpt-6: step = 5000
ckpt-7: step = 6000


In [27]:
from pathlib import Path

config_path = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_stage5.config"
)

text = config_path.read_text()

print("=== PEMERIKSAAN KONFIGURASI STAGE 5 ===")

for i, line in enumerate(text.splitlines(), start=1):
    if any(key in line for key in [
        "num_steps:",
        "batch_size:",
        "fine_tune_checkpoint:",
        "input_path:",
        "label_map_path:",
        "num_classes:"
    ]):
        print(f"{i}: {line.strip()}")

=== PEMERIKSAAN KONFIGURASI STAGE 5 ===
3: num_classes: 8
135: batch_size: 1
165: fine_tune_checkpoint: "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0"
166: num_steps: 5945
175: label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
177: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record"
185: label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
189: input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"


In [28]:
# Konfigurasi evaluasi pada test set

from pathlib import Path
import re

source_config = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_stage5.config"
)

eval_config = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

test_record = (
    "/home/rzangef/SSD-PPE/dataset/"
    "tfrecord_yolo_split/test.record"
)

label_map = (
    "/home/rzangef/SSD-PPE/dataset/"
    "tfrecord_yolo_split/label_map.pbtxt"
)

assert source_config.exists(), f"Config tidak ditemukan: {source_config}"
assert Path(test_record).exists(), f"Test record tidak ditemukan: {test_record}"
assert Path(label_map).exists(), f"Label map tidak ditemukan: {label_map}"

config = source_config.read_text()

# Ganti hanya input_path pada blok eval_config.
marker = "eval_config {"
start = config.find(marker)
assert start >= 0, "Blok eval_config tidak ditemukan."

next_block = config.find("\n}", start)
assert next_block >= 0, "Akhir blok eval_config tidak ditemukan."

eval_block = config[start:next_block + 2]
eval_block_new = re.sub(
    r'input_path:\s*"[^"]*"',
    f'input_path: "{test_record}"',
    eval_block,
    count=1
)
eval_block_new = re.sub(
    r'label_map_path:\s*"[^"]*"',
    f'label_map_path: "{label_map}"',
    eval_block_new,
    count=1
)

config = config[:start] + eval_block_new + config[next_block + 2:]

eval_config.write_text(config)

print("Konfigurasi evaluasi dibuat:", eval_config)
print("Test record:", test_record)
print("Label map:", label_map)
print("Konfigurasi training asli tetap tidak diubah.")

Konfigurasi evaluasi dibuat: /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config
Test record: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record
Label map: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt
Konfigurasi training asli tetap tidak diubah.


In [29]:
from pathlib import Path

config_path = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

lines = config_path.read_text().splitlines()

print("=== BLOK EVALUASI ===")
inside_eval = False

for line in lines:
    if "eval_config {" in line:
        inside_eval = True

    if inside_eval:
        print(line)

    if inside_eval and line.strip() == "}":
        break

print("\n=== PATH TEST RECORD ===")
for i, line in enumerate(lines, start=1):
    if "test.record" in line:
        print(f"Baris {i}: {line.strip()}")

=== BLOK EVALUASI ===
eval_config {
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false
}

=== PATH TEST RECORD ===


In [30]:
from pathlib import Path

model_dir = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)

print("Checkpoint state:")
state = model_dir / "checkpoint"

if state.exists():
    print(state.read_text())
else:
    print("File checkpoint tidak ditemukan.")

print("\nckpt-7 tersedia:", (model_dir / "ckpt-7.index").exists())

Checkpoint state:
model_checkpoint_path: "ckpt-7"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_paths: "ckpt-2"
all_model_checkpoint_paths: "ckpt-3"
all_model_checkpoint_paths: "ckpt-4"
all_model_checkpoint_paths: "ckpt-5"
all_model_checkpoint_paths: "ckpt-6"
all_model_checkpoint_paths: "ckpt-7"
all_model_checkpoint_timestamps: 1792483216.0080063
all_model_checkpoint_timestamps: 1792483558.2438378
all_model_checkpoint_timestamps: 1792483861.868789
all_model_checkpoint_timestamps: 1792484159.868539
all_model_checkpoint_timestamps: 1792484458.692329
all_model_checkpoint_timestamps: 1792484758.62851
all_model_checkpoint_timestamps: 1792485066.0702882
last_preserved_timestamp: 1792483214.359389


ckpt-7 tersedia: True


In [31]:
from pathlib import Path

config_path = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

text = config_path.read_text()
lines = text.splitlines()

for i, line in enumerate(lines):
    if "eval_config {" in line or "eval_input_reader {" in line:
        print(f"\n=== Mulai baris {i + 1} ===")
        depth = 0
        started = False

        for j in range(i, len(lines)):
            current = lines[j]
            print(f"{j + 1}: {current}")

            depth += current.count("{") - current.count("}")
            if "{" in current:
                started = True

            if started and depth == 0:
                break


=== Mulai baris 180 ===
180: eval_config {
181:   metrics_set: "coco_detection_metrics"
182:   use_moving_averages: false
183: }

=== Mulai baris 184 ===
184: eval_input_reader {
185:   label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
186:   shuffle: false
187:   num_epochs: 1
188:   tf_record_input_reader {
189:     input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"
190:   }
191: }


In [1]:
# Perbaiki konfigurasi evaluasi
from pathlib import Path
import re

config_path = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

test_record = (
    "/home/rzangef/SSD-PPE/dataset/"
    "tfrecord_yolo_split/test.record"
)

label_map = (
    "/home/rzangef/SSD-PPE/dataset/"
    "tfrecord_yolo_split/label_map.pbtxt"
)

assert config_path.exists(), "Konfigurasi evaluasi tidak ditemukan."
assert Path(test_record).exists(), "test.record tidak ditemukan."
assert Path(label_map).exists(), "label_map.pbtxt tidak ditemukan."

config = config_path.read_text()

pattern = r"eval_input_reader\s*\{.*?\n\}"
match = re.search(pattern, config, flags=re.DOTALL)

assert match, "Blok eval_input_reader tidak ditemukan."

new_block = f'''eval_input_reader {{
  label_map_path: "{label_map}"
  shuffle: false
  num_epochs: 1
  tf_record_input_reader {{
    input_path: "{test_record}"
  }}
}}'''

config = config[:match.start()] + new_block + config[match.end():]
config_path.write_text(config)

print("Konfigurasi evaluasi berhasil diperbaiki.")
print("File:", config_path)
print("Input:", test_record)

Konfigurasi evaluasi berhasil diperbaiki.
File: /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config
Input: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record


In [2]:
# verifikasi sebelum jalankan evaluasi
from pathlib import Path

p = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

text = p.read_text()

assert "test.record" in text, "test.record belum ditemukan."
assert "val.record" not in text, (
    "Masih ada referensi val.record dalam konfigurasi evaluasi."
)

print("Verifikasi berhasil: konfigurasi menggunakan test.record.")

Verifikasi berhasil: konfigurasi menggunakan test.record.


In [4]:
# Buat konfigurasi untuk data validasi
text = VAL_CONFIG.read_text()

for block_name in ["eval_config", "eval_input_reader"]:
    match = re.search(rf"\b{block_name}\s*\{{", text)
    if not match:
        print(f"\nBlok {block_name} tidak ditemukan.")
        continue

    start = match.start()
    open_brace = text.find("{", start)
    depth = 0

    for i in range(open_brace, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                print(f"\n=== {block_name} ===")
                print(text[start:i + 1])
                break

NameError: name 'VAL_CONFIG' is not defined

In [5]:
# Buat konfigurasi validasi
import re
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_stage5.config"
VAL_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
VAL_RECORD = PROJECT / "dataset/tfrecord_yolo_split/val.record"
LABEL_MAP = PROJECT / "dataset/tfrecord_yolo_split/label_map.pbtxt"

# Pastikan semua file sumber tersedia
for path in [CONFIG, VAL_RECORD, LABEL_MAP]:
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

def replace_proto_block(text, block_name, update_func):
    match = re.search(rf"\b{re.escape(block_name)}\s*\{{", text)
    if not match:
        raise ValueError(f"Blok {block_name} tidak ditemukan.")

    open_brace = text.find("{", match.start())
    depth = 0

    for i in range(open_brace, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                old_block = text[match.start():i + 1]
                new_block = update_func(old_block)
                return text[:match.start()] + new_block + text[i + 1:]

    raise ValueError(f"Blok {block_name} tidak tertutup dengan benar.")

def update_eval_reader(block):
    # Ganti label map
    block, n1 = re.subn(
        r'(?m)^(\s*label_map_path\s*:\s*)"[^"]*"',
        lambda m: m.group(1) + f'"{LABEL_MAP}"',
        block,
        count=1,
    )
    if n1 != 1:
        raise ValueError("label_map_path tidak ditemukan.")

    # Ganti input record menjadi data validasi
    block, n2 = re.subn(
        r'(?m)^(\s*input_path\s*:\s*)"[^"]*"',
        lambda m: m.group(1) + f'"{VAL_RECORD}"',
        block,
        count=1,
    )
    if n2 != 1:
        raise ValueError("input_path tidak ditemukan di eval_input_reader.")

    # Pastikan shuffle false
    if re.search(r"(?m)^\s*shuffle\s*:", block):
        block = re.sub(
            r"(?m)^(\s*shuffle\s*:\s*).*$",
            r"\g<1>false",
            block,
        )
    else:
        block = block[:-1] + "\n  shuffle: false\n}"

    # Pastikan hanya satu kali pembacaan dataset
    if re.search(r"(?m)^\s*num_epochs\s*:", block):
        block = re.sub(
            r"(?m)^(\s*num_epochs\s*:\s*).*$",
            r"\g<1>1",
            block,
        )
    else:
        block = block[:-1] + "\n  num_epochs: 1\n}"

    return block

def update_eval_config(block):
    if re.search(r"(?m)^\s*num_examples\s*:", block):
        block = re.sub(
            r"(?m)^(\s*num_examples\s*:\s*).*$",
            r"\g<1>170",
            block,
        )
    else:
        block = block[:-1] + "\n  num_examples: 170\n}"

    return block

# Baca konfigurasi training dan buat salinan di memori
config_text = CONFIG.read_text()

# Ubah blok evaluasi pada salinan
config_text = replace_proto_block(
    config_text, "eval_input_reader", update_eval_reader
)
config_text = replace_proto_block(
    config_text, "eval_config", update_eval_config
)

# Simpan sebagai file baru
VAL_CONFIG.write_text(config_text)

print("Konfigurasi validasi berhasil dibuat.")
print("Lokasi:", VAL_CONFIG)
print("File tersedia:", VAL_CONFIG.exists())
print("Konfigurasi training asli tetap:", CONFIG.exists())

Konfigurasi validasi berhasil dibuat.
Lokasi: /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_val.config
File tersedia: True
Konfigurasi training asli tetap: True


In [6]:
# verifikasi kongurasi validasi
print("=== KONFIGURASI VALIDASI ===")

text = VAL_CONFIG.read_text()

for block_name in ["eval_config", "eval_input_reader"]:
    match = re.search(rf"\b{block_name}\s*\{{", text)
    start = match.start()
    open_brace = text.find("{", start)
    depth = 0

    for i in range(open_brace, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                print(f"\n{block_name}:")
                print(text[start:i + 1])
                break

=== KONFIGURASI VALIDASI ===

eval_config:
eval_config {
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false

  num_examples: 170
}

eval_input_reader:
eval_input_reader {
  label_map_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"
  shuffle: false
  num_epochs: 1
  tf_record_input_reader {
    input_path: "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"
  }
}


In [7]:
# Jalankan evaluasi
from pathlib import Path
import subprocess
import os

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH_DIR = PROJECT / "object-detection-api/research"
SCRIPT = RESEARCH_DIR / "object_detection/model_main_tf2.py"

VAL_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
MODEL_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
EVAL_DIR = PROJECT / "evaluation/ssd_yolo_split_val"
LOG_FILE = PROJECT / "logs/ssd_stage5_validation.log"
WRAPPER = PROJECT / "run_ssd_validation_wrapper.py"

# Pastikan prasyarat tersedia
for path in [PYTHON, SCRIPT, VAL_CONFIG, MODEL_DIR / "checkpoint"]:
    if not path.exists():
        raise FileNotFoundError(f"File atau folder tidak ditemukan: {path}")

# Periksa checkpoint terakhir
print("=== CHECKPOINT STATE ===")
print((MODEL_DIR / "checkpoint").read_text())

# Siapkan folder output
EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

# Wrapper kompatibilitas TensorFlow Object Detection API
wrapper_code = r'''
import sys, runpy, tensorflow as tf
from tensorflow.python.ops import control_flow_ops

if not hasattr(control_flow_ops, "cond"):
    control_flow_ops.cond = tf.cond

script = sys.argv[1]
sys.argv = [script] + sys.argv[2:]
print("Temporary compatibility alias enabled:",
      hasattr(control_flow_ops, "cond"), flush=True)
runpy.run_path(script, run_name="__main__")
'''

WRAPPER.write_text(wrapper_code)

cmd = [
    str(PYTHON),
    str(WRAPPER),
    str(SCRIPT),
    f"--pipeline_config_path={VAL_CONFIG}",
    f"--model_dir={EVAL_DIR}",
    f"--checkpoint_dir={MODEL_DIR}",
    "--eval_timeout=0",
]

env = os.environ.copy()
paths = [str(RESEARCH_DIR), str(RESEARCH_DIR / "slim")]
if env.get("PYTHONPATH"):
    paths.append(env["PYTHONPATH"])
env["PYTHONPATH"] = os.pathsep.join(paths)

with open(LOG_FILE, "w") as log:
    process = subprocess.Popen(
        cmd,
        cwd=str(RESEARCH_DIR),
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("\nEvaluasi validasi diluncurkan.")
print("PID       :", process.pid)
print("Konfigurasi:", VAL_CONFIG)
print("Checkpoint:", MODEL_DIR)
print("Output    :", EVAL_DIR)
print("Log       :", LOG_FILE)

=== CHECKPOINT STATE ===
model_checkpoint_path: "ckpt-7"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_paths: "ckpt-2"
all_model_checkpoint_paths: "ckpt-3"
all_model_checkpoint_paths: "ckpt-4"
all_model_checkpoint_paths: "ckpt-5"
all_model_checkpoint_paths: "ckpt-6"
all_model_checkpoint_paths: "ckpt-7"
all_model_checkpoint_timestamps: 1792483216.0080063
all_model_checkpoint_timestamps: 1792483558.2438378
all_model_checkpoint_timestamps: 1792483861.868789
all_model_checkpoint_timestamps: 1792484159.868539
all_model_checkpoint_timestamps: 1792484458.692329
all_model_checkpoint_timestamps: 1792484758.62851
all_model_checkpoint_timestamps: 1792485066.0702882
last_preserved_timestamp: 1792483214.359389


Evaluasi validasi diluncurkan.
PID       : 1378
Konfigurasi: /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_val.config
Checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split
Output    : /home/rzangef/SSD-PPE/evaluation/ssd_yolo_split_val
Log

In [8]:
# Cek evaluasi apakah berjalan
from pathlib import Path

LOG_FILE = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_validation.log"
)

if LOG_FILE.exists():
    log_text = LOG_FILE.read_text(errors="replace")

    print("=== 12.000 KARAKTER TERAKHIR LOG ===")
    print(log_text[-12000:])
else:
    print("File log belum ditemukan:", LOG_FILE)

=== 12.000 KARAKTER TERAKHIR LOG ===
2026-10-21 08:20:37.626061: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 08:20:38.356505: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-21 08:20:38.357044: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-21 08:20:38.499282: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-21 08:20:38.799611: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 08:20:38.802255: I tensorflow

In [9]:
import subprocess

result = subprocess.run(
    ["bash", "-lc", "ps -p 1378 -o pid,stat,etime,cmd"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout.strip()
      else "Proses PID 1378 sudah tidak berjalan.")

    PID STAT     ELAPSED CMD
   1378 Ssl        01:29 /home/rzangef/SSD-PPE/.venv/bin/python /home/rzangef/SSD-PPE/run_ssd_validation_wrapper.py /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_main_tf2.py --pipeline_config_path=/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_val.config --model_dir=/home/rzangef/SSD-PPE/evaluation/ssd_yolo_split_val --checkpoint_dir=/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split --eval_timeout=0



In [10]:
from pathlib import Path

LOG_FILE = Path(
    "/home/rzangef/SSD-PPE/logs/ssd_stage5_validation.log"
)

if LOG_FILE.exists():
    log_text = LOG_FILE.read_text(errors="replace")

    print("Ukuran log:", len(log_text), "karakter")
    print("\n=== LOG EVALUASI TERBARU ===")
    print(log_text[-15000:] if log_text else "Log masih kosong.")
else:
    print("File log belum ditemukan.")

Ukuran log: 15279 karakter

=== LOG EVALUASI TERBARU ===
g to register factory for plugin cuDNN when one has already been registered
2026-10-21 08:20:38.357044: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-21 08:20:38.499282: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-21 08:20:38.799611: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 08:20:38.802255: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate com

In [11]:
# cek status evaluasi dan log training
from pathlib import Path
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")

print("=== STATUS PROSES EVALUASI ===")
result = subprocess.run(
    ["bash", "-lc", "ps -p 1378 -o pid,stat,etime,cmd"],
    capture_output=True,
    text=True
)
print(result.stdout.strip() or "Proses evaluasi sudah berhenti.")

print("\n=== AKHIR LOG TRAINING ===")
train_log = PROJECT / "logs/ssd_stage5_training.log"

if train_log.exists():
    print(train_log.read_text(errors="replace")[-7000:])
else:
    print("Log training tidak ditemukan.")

print("\n=== FILE OUTPUT EVALUASI ===")
eval_dir = PROJECT / "evaluation/ssd_yolo_split_val"

if eval_dir.exists():
    for path in sorted(eval_dir.rglob("*")):
        if path.is_file():
            print(path.relative_to(PROJECT), path.stat().st_size, "bytes")
else:
    print("Folder output evaluasi tidak ditemukan.")

=== STATUS PROSES EVALUASI ===
PID STAT     ELAPSED CMD
   1378 Zs         20:07 [model_main_tf2.] <defunct>

=== AKHIR LOG TRAINING ===
0.15335667,
 'Loss/total_loss': 0.7590138,
 'learning_rate': 2.8454895e-05}
INFO:tensorflow:Step 4900 per-step time 0.305s
I1020 15:25:26.879413 140488109375488 model_lib_v2.py:705] Step 4900 per-step time 0.305s
INFO:tensorflow:{'Loss/classification_loss': 0.8055403,
 'Loss/localization_loss': 0.5134126,
 'Loss/regularization_loss': 0.15335642,
 'Loss/total_loss': 1.4723094,
 'learning_rate': 2.3832117e-05}
I1020 15:25:26.879893 140488109375488 model_lib_v2.py:708] {'Loss/classification_loss': 0.8055403,
 'Loss/localization_loss': 0.5134126,
 'Loss/regularization_loss': 0.15335642,
 'Loss/total_loss': 1.4723094,
 'learning_rate': 2.3832117e-05}
INFO:tensorflow:Step 5000 per-step time 0.306s
I1020 15:25:57.467384 140488109375488 model_lib_v2.py:705] Step 5000 per-step time 0.306s
INFO:tensorflow:{'Loss/classification_loss': 0.625382,
 'Loss/localizati

In [12]:

from pathlib import Path
import sys

PROJECT = Path("/home/rzangef/SSD-PPE")
EVENT_DIR = PROJECT / "evaluation/ssd_yolo_split_val/eval"

# Pastikan TensorFlow berasal dari environment SSD
PYTHON = PROJECT / ".venv/bin/python"
print("Python environment:", PYTHON)
print("Folder evaluasi tersedia:", EVENT_DIR.exists())

event_files = sorted(EVENT_DIR.glob("events.out.tfevents.*"))

print("\nJumlah file event:", len(event_files))
for f in event_files:
    print(f.name, "|", round(f.stat().st_size / (1024 * 1024), 2), "MB")

if not event_files:
    print("\nFile event belum ditemukan.")
else:
    try:
        from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

        for event_file in event_files:
            print("\n=== METRIK DARI FILE EVENT ===")
            print(event_file.name)

            ea = EventAccumulator(str(event_file))
            ea.Reload()

            tags = ea.Tags()
            scalar_tags = tags.get("scalars", [])

            if not scalar_tags:
                print("Tidak ada scalar tags. Tags yang tersedia:", tags)
            else:
                for tag in scalar_tags:
                    values = ea.Scalars(tag)
                    if values:
                        last = values[-1]
                        print(
                            f"{tag}: {last.value:.6f}"
                            f" | step={last.step}"
                        )

    except Exception as e:
        print("Gagal membaca event:", repr(e))

Python environment: /home/rzangef/SSD-PPE/.venv/bin/python
Folder evaluasi tersedia: True

Jumlah file event: 1
events.out.tfevents.1792545663.LAPTOP-ODRK022O.1378.0.v2 | 2.22 MB

=== METRIK DARI FILE EVENT ===
events.out.tfevents.1792545663.LAPTOP-ODRK022O.1378.0.v2


2026-10-21 08:47:15.468131: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 08:47:16.166898: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-21 08:47:16.167257: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-21 08:47:16.307082: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-21 08:47:16.609370: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-21 08:47:16.613344: I tensorflow/core/platform/cpu_feature_guard.cc:1

Tidak ada scalar tags. Tags yang tersedia: {'images': [], 'audio': [], 'histograms': [], 'scalars': [], 'distributions': [], 'tensors': ['eval_side_by_side_0_0', 'eval_side_by_side_1_0', 'eval_side_by_side_2_0', 'eval_side_by_side_3_0', 'eval_side_by_side_4_0', 'eval_side_by_side_5_0', 'eval_side_by_side_6_0', 'eval_side_by_side_7_0', 'eval_side_by_side_8_0', 'eval_side_by_side_9_0', 'DetectionBoxes_Precision/mAP', 'DetectionBoxes_Precision/mAP@.50IOU', 'DetectionBoxes_Precision/mAP@.75IOU', 'DetectionBoxes_Precision/mAP (small)', 'DetectionBoxes_Precision/mAP (medium)', 'DetectionBoxes_Precision/mAP (large)', 'DetectionBoxes_Recall/AR@1', 'DetectionBoxes_Recall/AR@10', 'DetectionBoxes_Recall/AR@100', 'DetectionBoxes_Recall/AR@100 (small)', 'DetectionBoxes_Recall/AR@100 (medium)', 'DetectionBoxes_Recall/AR@100 (large)', 'Loss/localization_loss', 'Loss/classification_loss', 'Loss/regularization_loss', 'Loss/total_loss'], 'graph': False, 'meta_graph': False, 'run_metadata': []}


In [13]:

from pathlib import Path
import tensorflow as tf
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

EVENT_DIR = Path(
    "/home/rzangef/SSD-PPE/evaluation/ssd_yolo_split_val/eval"
)

event_files = sorted(EVENT_DIR.glob("events.out.tfevents.*"))

if not event_files:
    print("File event tidak ditemukan.")
else:
    for event_file in event_files:
        print("File:", event_file.name)

        ea = EventAccumulator(str(event_file))
        ea.Reload()

        tags = ea.Tags().get("tensors", [])

        target_tags = [
            tag for tag in tags
            if tag.startswith("DetectionBoxes_Precision/")
            or tag.startswith("DetectionBoxes_Recall/")
            or tag.startswith("Loss/")
        ]

        for tag in target_tags:
            tensor_events = ea.Tensors(tag)

            if not tensor_events:
                continue

            last = tensor_events[-1]
            array = tf.make_ndarray(last.tensor_proto)

            print(f"\n{tag}")
            print("Step:", last.step)
            print("Nilai:", array)

        print("\n=== SELESAI MEMBACA METRIK ===")

File: events.out.tfevents.1792545663.LAPTOP-ODRK022O.1378.0.v2

DetectionBoxes_Precision/mAP
Step: 6000
Nilai: 0.042923104

DetectionBoxes_Precision/mAP@.50IOU
Step: 6000
Nilai: 0.120706476

DetectionBoxes_Precision/mAP@.75IOU
Step: 6000
Nilai: 0.01967437

DetectionBoxes_Precision/mAP (small)
Step: 6000
Nilai: 0.0

DetectionBoxes_Precision/mAP (medium)
Step: 6000
Nilai: 0.025933577

DetectionBoxes_Precision/mAP (large)
Step: 6000
Nilai: 0.06236745

DetectionBoxes_Recall/AR@1
Step: 6000
Nilai: 0.07447062

DetectionBoxes_Recall/AR@10
Step: 6000
Nilai: 0.17735974

DetectionBoxes_Recall/AR@100
Step: 6000
Nilai: 0.20586808

DetectionBoxes_Recall/AR@100 (small)
Step: 6000
Nilai: 0.0

DetectionBoxes_Recall/AR@100 (medium)
Step: 6000
Nilai: 0.17959286

DetectionBoxes_Recall/AR@100 (large)
Step: 6000
Nilai: 0.31106785

Loss/localization_loss
Step: 6000
Nilai: 0.5057597

Loss/classification_loss
Step: 6000
Nilai: 0.58791834

Loss/regularization_loss
Step: 6000
Nilai: 0.15335563

Loss/total_loss
